# 00 - Setup & Data Download

This notebook:
1. Installs required libraries
2. Sets up our database schemas
3. Downloads the Olist dataset from Kaggle into our storage

In [0]:
#%pip install kaggle

In [0]:
dbutils.widgets.text("kaggle_username", "","Kaggle Username")
dbutils.widgets.text("kaggle_key", "", "Kaggle API key")

In [0]:
import os

kaggle_username = dbutils.widgets.get("kaggle_username")
kaggle_key = dbutils.widgets.get("kaggle_key")

os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"] = kaggle_key

print("Kaggle credentials configured!")

## Download the Olist Dataset

In [0]:
%sql
-- Create a schema to hold our raw data volume
CREATE SCHEMA IF NOT EXISTS workspace.retail_landing;

-- Create a Volume (a managed folder for raw files)
CREATE VOLUME IF NOT EXISTS workspace.retail_landing.raw_data;

In [0]:
import os
import subprocess
import shutil

volume_path = "/Volumes/workspace/retail_landing/raw_data"

# Step 1: Download from Kaggle to /tmp
os.makedirs("/tmp/olist", exist_ok=True)

print("Downloading from Kaggle...")
subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "olistbr/brazilian-ecommerce",
    "-p", "/tmp/olist",
    "--unzip"
], check=True)
print("Download complete!\n")

# Step 2: Copy using Python shutil (not dbutils)
print("Copying to permanent storage...")
for filename in os.listdir("/tmp/olist"):
    src = f"/tmp/olist/{filename}"
    dst = f"{volume_path}/{filename}"
    shutil.copy2(src, dst)
    print(f"✅ {filename}")

print("\n🎉 All files safely stored in Unity Catalog Volume!")

In [0]:
print("📦 Files in Unity Catalog Volume:\n")

files = os.listdir(volume_path)
for f in sorted(files):
    size_mb = round(os.path.getsize(f"{volume_path}/{f}") / 1024 / 1024, 2)
    print(f"  📄 {f} — {size_mb} MB")

print(f"\n✅ Total: {len(files)} files")